# Qwen3-VL — Retrieval-Based (Dynamic) Few-Shot for AVM Classification

**Author:** Suhani Shokeen  **Experiment:** the 3rd rung of the ladder.

| Condition | In-context examples | Status |
|---|---|---|
| Zero-shot (control) | none | done (other notebooks) |
| Static few-shot | same fixed 10 for every query | done (teammate) |
| **Retrieval few-shot (this notebook)** | **per-query: the k most visually similar labeled images** | this notebook |

**Idea.** Instead of showing the model the *same* 10 examples each time, we embed every
image with CLIP, and for each query image we retrieve its `k` nearest neighbours (by image
similarity) and use *those* as the few-shot examples. If visual similarity tracks the AVM
label, the demonstrations are more relevant per query and accuracy should improve.

Everything else (system prompt, taxonomy, metrics) is held identical to the other two runs
so the only thing that changes is *which examples appear* — that isolates the retrieval effect.

> **Compute heads-up:** the single most expensive cell is the **VLM inference loop**
> (§7). Each prompt now carries `k` example images + 1 query image, so it uses far more
> VRAM and time than zero-shot. See §9 for the full breakdown and OOM mitigations.

## 1. Setup  (Colab)

**Before running:** go to **Runtime -> Change runtime type -> Hardware accelerator = GPU**.
- Pick an **A100** (Colab Pro) to run the 8B model and match the teammate's static run.
- On a free **T4 (16 GB)** the 8B will OOM; in section 4 set `MODEL_ID` to the 2B and lower `K_SHOT`.

The cells below install everything in-notebook, so there is no separate environment to set up.

In [ ]:
# Confirm a GPU is attached. If this errors, you are on a CPU runtime -> switch it (see above).
!nvidia-smi

In [ ]:
# --- Install everything the notebook needs (Colab-ready) -------------------------------
# Colab already ships torch (CUDA build), numpy, pandas, pillow. We only add the rest.
# transformers is installed FROM SOURCE because the Qwen3-VL classes are not in a release yet.
!pip install -q -U "git+https://github.com/huggingface/transformers"
!pip install -q -U accelerate datasets
# If you ever run this OFF Colab, also: pip install pillow pandas numpy
print("Installs complete. (If a later import fails with a version error, do Runtime -> Restart, then re-run.)")

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# 2B fits a free T4 (16 GB). The 8B needs ~A100 and will OOM on a T4.
# Whichever you pick, compare against the SAME-size runs (see note below the model load).
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
# MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"   # only on an A100 runtime

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,        # T4 (Turing) -> fp16. bf16 is an A100/Ampere feature; use it only there.
    low_cpu_mem_usage=True,     # stream weights in -> lower host-RAM peak during load
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Loaded", MODEL_ID, "on", model.device)

> **Fair-comparison note.** This run uses the **2B** model (T4-friendly). For the §10 comparison to
> be apples-to-apples, compare it against **2B** versions of the other two runs — i.e. the existing
> 2B zero-shot results, and a 2B static few-shot run. Don't compare a 2B retrieval run against the
> teammate's **8B** static numbers; model size, not the retrieval method, would explain the gap.
> If your teammate only has 8B static numbers, ask them to re-run static on the 2B (same notebook,
> just the 2B model) so all three rungs share a model size.

In [ ]:
from PIL import Image
from IPython.display import display

DISPLAY_MAX_SIZE = 300

def _show(img_or_path):
    img = Image.open(img_or_path) if isinstance(img_or_path, str) else img_or_path
    preview = img.copy(); preview.thumbnail((DISPLAY_MAX_SIZE, DISPLAY_MAX_SIZE)); display(preview)

## 2. Dataset  (same 266 ESA-Hubble rows as the other notebooks)

In [ ]:
### Do not rerun unless the runtime restarted. ###
from datasets import load_dataset
from itertools import islice
from PIL import Image

# Hubble images can be 90+ megapixels. Disable PIL's "decompression bomb" guard, and DOWNSIZE
# each image the moment it streams in. If we instead did `rows = list(islice(...))`, every image
# would be decoded to full resolution and held in RAM at once -> tens of GB -> kernel OOM/crash.
Image.MAX_IMAGE_PIXELS = None
CACHE_MAX_PX = 1024   # we only ever use 512 (examples) / 768 (query); CLIP uses 224. 1024 loses nothing.

DATASET_ID = "Supermaxman/esa-hubble"
dataset_stream = load_dataset(DATASET_ID, split="train", streaming=True)

N = 266
rows, images_full, names = [], [], []
for i, r in enumerate(islice(dataset_stream, N)):
    img = r["image"].convert("RGB")
    img.thumbnail((CACHE_MAX_PX, CACHE_MAX_PX))                 # shrink now; the full-res original is freed
    images_full.append(img)
    names.append((r.get("Name") or "").strip())                # for same-object exclusion (section 6b)
    rows.append({k: v for k, v in r.items() if k != "image"})  # keep metadata, drop the heavy image
    if (i + 1) % 25 == 0:
        print(f"  streamed {i + 1}/{N}")

print("Loaded rows:", len(rows), "| cached downsized images:", len(images_full))
print("Columns:", list(rows[0].keys()))

## 3. Reference files — AVM taxonomy + ground-truth codes

In [ ]:
import urllib.request

PROJECT_RAW_BASE_URL = "https://raw.githubusercontent.com/QihanQG/ECS-189G-final-project/main"
AVM_REFERENCE_URL  = f"{PROJECT_RAW_BASE_URL}/AVM_Reference"
TRUE_AVM_CODES_URL = f"{PROJECT_RAW_BASE_URL}/TRUE_AVM_CODES"

def load_text_from_url(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        return r.read().decode("utf-8")

def parse_true_avm_codes(text):
    labels_by_row = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        row_text, codes_text = line.split(":", 1)
        labels_by_row[int(row_text.strip())] = [c.strip() for c in codes_text.split(",") if c.strip()]
    assert sorted(labels_by_row) == list(range(len(labels_by_row))), "TRUE_AVM_CODES rows missing/out of order"
    return [labels_by_row[i] for i in range(len(labels_by_row))]

avm_reference_text = load_text_from_url(AVM_REFERENCE_URL)
TRUE_AVM_CODES     = parse_true_avm_codes(load_text_from_url(TRUE_AVM_CODES_URL))

# The AVM_Reference file is literally Python that defines AVM_SCALE and AVM_TAXONOMY dicts.
# exec it so we can turn a code like "C.5.1.7" into a human-readable gloss for the prompt.
_avm_ns = {}
exec(avm_reference_text, _avm_ns)
AVM_SCALE    = _avm_ns["AVM_SCALE"]
AVM_TAXONOMY = _avm_ns["AVM_TAXONOMY"]

print(f"AVM reference: {len(avm_reference_text):,} chars | ground-truth rows: {len(TRUE_AVM_CODES)}")
print("Row 0 true code(s):", TRUE_AVM_CODES[0])

## 4. Config — all the knobs in one place

This also **mounts Google Drive** and writes all predictions there, so a Colab disconnect does
not lose your results. The folder `MyDrive/ECS189G_RAG/` is created automatically; drop your
teammate's static prediction file into that same folder to make the §10 comparison populate.

In [ ]:
from pathlib import Path

# ---- persist outputs to Google Drive (survives Colab disconnects) ----
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/ECS189G_RAG")
except Exception:
    OUTPUT_DIR = Path(".")        # not on Colab -> just use the current directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs ->", OUTPUT_DIR.resolve())

# ---- few-shot retrieval settings ----
K_SHOT             = 10      # examples retrieved per query (match teammate's static count)
EXAMPLE_MAX_PX     = 512     # downsize retrieved example images (match teammate)
QUERY_MAX_PX       = 768     # downsize the query image       (match teammate)
EXCLUDE_SAME_OBJECT = True   # drop neighbours sharing the query's `Name` (anti-leakage, see §6)
ADD_GLOSS          = True    # include human-readable gloss next to each example's code ("RAG text")

# ---- retrieval embedding model ----
CLIP_MODEL_ID      = "openai/clip-vit-base-patch32"

# ---- evaluation / bookkeeping ----
RUN_NAME           = "retrieval_v1"
PREDICTIONS_OUTPUT_PATH = OUTPUT_DIR / f"qwen_predicted_avm_codes_{RUN_NAME}.txt"
NUM_SAMPLES_TO_RUN = 266     # evaluate every image (leave-one-out, see §6)

# Prediction files from the OTHER two runs, for the §10 comparison.
# Upload these into OUTPUT_DIR (MyDrive/ECS189G_RAG/) to include them.
ZEROSHOT_PRED_PATH = OUTPUT_DIR / "qwen_predicted_avm_codes_strict_codes_v1.txt"  # control
STATIC_PRED_PATH   = OUTPUT_DIR / "qwen_predicted_avm_codes_static_fewshot.txt"   # teammate's static run

# The 10 rows the teammate used as fixed static examples (for reference / fair test-set alignment).
STATIC_FEWSHOT_INDICES = [1, 3, 4, 12, 15, 16, 50, 57, 74, 163]

print("Config loaded.")

## 5. Image preprocessing — resize to a max side (keeps aspect ratio)

In [ ]:
def to_rgb(image):
    if isinstance(image, str):
        return Image.open(image).convert("RGB")
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    return Image.fromarray(image).convert("RGB")

# Some Hubble images are 90+ megapixels. Disable PIL's "decompression bomb" guard so they
# load, and (critically) NEVER keep them at full resolution -- that exhausts Colab RAM and
# crashes the kernel. We cache a downsized copy instead.
Image.MAX_IMAGE_PIXELS = None

CACHE_MAX_PX = 1024   # cache at <=1024px. We only ever use 512 (examples) / 768 (query),
                      # and CLIP downsamples to 224 internally, so this loses nothing useful.

def resize_max_side(image, max_px):
    '''Downscale so the longest side <= max_px. Never upscales. Returns a copy.'''
    img = to_rgb(image).copy()
    img.thumbnail((max_px, max_px))   # PIL keeps aspect ratio, only shrinks
    return img

# NOTE: images_full and names were already built (downsized) while streaming in section 2 --
# we do NOT rebuild them from rows here, because section 2 dropped the heavy "image" key.
# These helpers are used later to make the 512px example / 768px query copies on demand.
print("Have", len(images_full), "cached images;",
      "min/max side example:", min(images_full[0].size), "/", max(images_full[0].size),
      "| name[0]:", repr(names[0]))

## 6. Build the retrieval index (CLIP embeddings)  ⏱ *one-time, moderate compute*

We embed all 266 images with CLIP **once** and cache the matrix. Retrieval is then just a
cosine-similarity lookup. This cell is the 2nd-heaviest in the notebook (after inference),
but it only runs once — re-running later cells does **not** recompute it.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(
    "cuda" if torch.cuda.is_available() else "cpu").eval()
clip_proc  = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)

@torch.no_grad()
def embed_images(pil_images, batch_size=16):
    feats = []
    dev = clip_model.device
    for i in range(0, len(pil_images), batch_size):
        batch = pil_images[i:i + batch_size]
        inputs = clip_proc(images=batch, return_tensors="pt").to(dev)
        out = clip_model.get_image_features(**inputs)
        # transformers-from-source sometimes returns an output object instead of a tensor;
        # pull the projected image embedding (or pooled features) out of it.
        if isinstance(out, torch.Tensor):
            emb = out
        elif getattr(out, "image_embeds", None) is not None:
            emb = out.image_embeds
        else:
            emb = out.pooler_output
        emb = F.normalize(emb, dim=-1)            # unit vectors -> cosine = dot product
        feats.append(emb.cpu())
        print(f"  embedded {min(i + batch_size, len(pil_images))}/{len(pil_images)}", end="\r")
    return torch.cat(feats, dim=0)

IMAGE_EMBEDDINGS = embed_images(images_full)      # (266, 512), L2-normalized
print("\nEmbedding matrix:", tuple(IMAGE_EMBEDDINGS.shape))

### 6b. Retrieval function

For a query row `i` we score every other row by cosine similarity and return the top-`k`.

Two anti-leakage rules make the comparison defensible:

* **Leave-one-out** — the query is never retrieved as its own example.
* **Same-object exclusion** (`EXCLUDE_SAME_OBJECT`) — this dataset contains sibling frames of
  the *same* object (e.g. several rows are all the same planetary nebula). Without this, retrieval
  would pull a near-identical twin carrying the identical label and trivially inflate the score.
  We drop any candidate sharing the query's `Name`.

In [ ]:
import torch

@torch.no_grad()
def retrieve_neighbors(query_idx, k=K_SHOT, exclude_same_object=EXCLUDE_SAME_OBJECT):
    sims = IMAGE_EMBEDDINGS @ IMAGE_EMBEDDINGS[query_idx]   # (266,) cosine similarities
    sims = sims.clone()
    sims[query_idx] = -1e9                                  # leave-one-out
    if exclude_same_object and names[query_idx]:
        for j, nm in enumerate(names):
            if nm and nm == names[query_idx]:
                sims[j] = -1e9                              # drop same-object siblings
    topk = torch.topk(sims, k).indices.tolist()
    return topk

# sanity check on one query
demo = retrieve_neighbors(0, k=5)
print("Query row 0 nearest neighbours:", demo)
print("Their true codes:", [TRUE_AVM_CODES[j] for j in demo])

## 7. Build the dynamic few-shot prompt

The conversation handed to Qwen is:

1. **system** — the full AVM taxonomy + task instructions (identical to the other two runs).
2. **k demonstration turns** — for each retrieved neighbour: a *user* turn with its (512px) image,
   and an *assistant* turn giving the correct AVM code(s). This is the "RAG text": the retrieved
   evidence rendered into the prompt. With `ADD_GLOSS=True` the human-readable meaning is appended
   to the example so the model also sees what the code *means*.
3. **final user turn** — the (768px) query image with the same instruction used in zero-shot.

In [ ]:
# ---- system prompt: identical to control/static for a fair comparison ----
avm_task_instructions = '''
You are an expert astronomical image classifier. Given an astronomical image, your task is to
identify the type(s) of object(s) or phenomenon(a) depicted and express each as an AVM 1.1 code.

AVM 1.1 Code Format: each code is <Scale>.<Taxonomy>.
Scale: A=Solar System, B=Milky Way, C=Local Universe, D=Early Universe, E=Unspecified.
Taxonomy: dot-separated numbers, e.g. B.4.1.3 -> Milky Way : Nebula : Type : Planetary;
C.5.1.1 -> Local Universe : Galaxy : Type : Spiral.

Instructions:
- Examine the image and identify all astronomical object types/phenomena present.
- For each, give the most specific matching AVM 1.1 code.
- An image may contain more than one type; if so, give one code per type, comma-separated.
- Output ONLY the AVM code(s), nothing else. No explanations, no labels.
Output format -> Single: B.4.1.3   Multiple: C.5.1.1, C.5.1.2
'''
avm_system_prompt = avm_reference_text + "\n\n" + avm_task_instructions

def code_to_gloss(code):
    '''Map e.g. C.5.1.7 to Local Universe : Galaxy : Type : Interacting. Best-effort; skips unknown parts.'''
    scale, *nums = code.split(".")
    parts = [AVM_SCALE.get(scale, scale)]
    for end in range(1, len(nums) + 1):
        key = ".".join(nums[:end])
        if key in AVM_TAXONOMY:
            parts.append(AVM_TAXONOMY[key])
    return " : ".join(parts)

def build_fewshot_messages(query_idx, neighbor_idxs):
    '''Assemble the system + demonstration + query conversation for one query image.'''
    messages = [{"role": "system", "content": [{"type": "text", "text": avm_system_prompt}]}]

    for j in neighbor_idxs:
        ex_img   = resize_max_side(images_full[j], EXAMPLE_MAX_PX)
        ex_codes = TRUE_AVM_CODES[j]
        answer   = ", ".join(ex_codes)
        if ADD_GLOSS:
            answer += "    # " + " | ".join(code_to_gloss(c) for c in ex_codes)
        messages.append({"role": "user", "content": [
            {"type": "image", "image": ex_img},
            {"type": "text",  "text": "Return only the AVM code or codes for this image."},
        ]})
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})

    query_img = resize_max_side(images_full[query_idx], QUERY_MAX_PX)
    messages.append({"role": "user", "content": [
        {"type": "image", "image": query_img},
        {"type": "text",  "text": "Return only the AVM code or codes for this image."},
    ]})
    return messages

# peek at the structure built for one query (text only, images omitted for readability)
_demo_msgs = build_fewshot_messages(0, retrieve_neighbors(0, k=2))
for m in _demo_msgs:
    txt = " ".join(c.get("text", "[image]") for c in m["content"])[:90]
    print(f"{m['role']:>9}: {txt}")

## 8. Batch inference   🔴 **HEAVIEST CELL — most compute / VRAM**

For every query we retrieve neighbours, build the prompt, and generate. Each prompt holds
`K_SHOT` example images + 1 query image, so this is ~10× the visual tokens of the zero-shot run.
Predictions are saved after every image so a crash/restart doesn't lose progress.

If you hit CUDA OOM: lower `K_SHOT`, lower `EXAMPLE_MAX_PX`, or switch `MODEL_ID` to the 2B.

**On a T4 with the 2B:** this should fit, but 11 images per prompt is the tight part. If you see
OOM, first drop `K_SHOT` to 6 (in §4), then `EXAMPLE_MAX_PX` to 384. A T4 is also slower, so the
full 266-image loop may take a while — predictions stream to Drive after each image, so a
disconnect/restart resumes safely. Keep the Colab tab active to avoid the idle timeout.

In [ ]:
import re
import torch

AVM_CODE_PATTERN = re.compile(r"\b[A-EX]\.\d+(?:\.\d+)*\b")

@torch.no_grad()
def generate_codes(messages, max_new_tokens=100):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

def extract_avm_codes(reply):
    codes = AVM_CODE_PATTERN.findall(reply)
    return codes if codes else [reply.replace("\n", " ").strip()]

prediction_lines = []
for row_index in range(NUM_SAMPLES_TO_RUN):
    print(f"Processing {row_index + 1}/{NUM_SAMPLES_TO_RUN}", end="\r")
    neighbors = retrieve_neighbors(row_index, k=K_SHOT)
    messages  = build_fewshot_messages(row_index, neighbors)
    reply     = generate_codes(messages, max_new_tokens=100)
    codes     = extract_avm_codes(reply)
    prediction_lines.append(f"{row_index}: {', '.join(codes)}")
    PREDICTIONS_OUTPUT_PATH.write_text("\n".join(prediction_lines) + "\n", encoding="utf-8")

print(f"\nSaved {len(prediction_lines)} predictions -> {PREDICTIONS_OUTPUT_PATH}")

### 8b. Inspect predictions

In [ ]:
print(PREDICTIONS_OUTPUT_PATH.resolve())
print(PREDICTIONS_OUTPUT_PATH.read_text(encoding="utf-8")[:1500])

## 9. Metrics  (identical functions to the control/static runs)

In [ ]:
import pandas as pd

def parse_prediction_file(path):
    preds = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or ":" not in line:
            continue
        idx, pred_text = line.split(":", 1)
        preds[int(idx.strip())] = AVM_CODE_PATTERN.findall(pred_text)
    return preds

def precision_recall_f1(predicted_codes, true_codes):
    p, t = set(predicted_codes), set(true_codes)
    if not p and not t: return 1.0, 1.0, 1.0
    if not p or not t:  return 0.0, 0.0, 0.0
    correct = len(p & t)
    prec = correct / len(p); rec = correct / len(t)
    f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    return prec, rec, f1

def code_ancestors(code):
    parts = code.split(".")
    return [".".join(parts[:e]) for e in range(1, len(parts) + 1)]

def expand_with_ancestors(codes):
    out = set()
    for c in codes: out.update(code_ancestors(c))
    return out

def evaluate(pred_path, true_codes=TRUE_AVM_CODES, restrict_to=None):
    '''Return a summary dict + per-row DataFrame. restrict_to = iterable of row indices to score.'''
    preds = parse_prediction_file(pred_path)
    idxs = sorted(set(preds) & set(range(len(true_codes))))
    if restrict_to is not None:
        idxs = [i for i in idxs if i in set(restrict_to)]
    recs = []
    for i in idxs:
        tc, pc = true_codes[i], preds.get(i, [])
        p, r, f = precision_recall_f1(pc, tc)
        hp, hr, hf = precision_recall_f1(expand_with_ancestors(pc), expand_with_ancestors(tc))
        recs.append(dict(row_index=i, predicted_codes=pc, true_codes=tc,
                         exact=set(pc) == set(tc), over=len(pc) > len(tc),
                         precision=p, recall=r, f1=f,
                         h_precision=hp, h_recall=hr, h_f1=hf))
    df = pd.DataFrame(recs)
    summary = dict(
        rows=len(df),
        exact_matches=int(df.exact.sum()), exact_rate=df.exact.mean(),
        macro_precision=df.precision.mean(), macro_recall=df.recall.mean(), macro_f1=df.f1.mean(),
        over_pred_rows=int(df.over.sum()),
        h_precision=df.h_precision.mean(), h_recall=df.h_recall.mean(), h_f1=df.h_f1.mean(),
    )
    return summary, df

summary, metrics_df = evaluate(PREDICTIONS_OUTPUT_PATH)
print("Metric 1 - Exact set match:",
      f"{summary['exact_matches']}/{summary['rows']} = {summary['exact_rate']:.4f}")
print("Metric 2 - Macro F1:        "
      f"P={summary['macro_precision']:.4f} R={summary['macro_recall']:.4f} F1={summary['macro_f1']:.4f}"
      f"  (over-prediction rows: {summary['over_pred_rows']})")
print("Metric 3 - Hierarchical F1: "
      f"P={summary['h_precision']:.4f} R={summary['h_recall']:.4f} F1={summary['h_f1']:.4f}")
metrics_df.head()

## 10. Comparison — zero-shot vs static vs retrieval

Scores all three runs on the **same common set of rows** (the intersection of indices present in
every prediction file) so the comparison is apples-to-apples. If a teammate's file is missing the
row is simply skipped with a note.

In [ ]:
def common_indices(*pred_paths):
    sets = []
    for p in pred_paths:
        if Path(p).exists():
            sets.append(set(parse_prediction_file(p)))
    return sorted(set.intersection(*sets)) if sets else []

runs = {
    "zero-shot (control)": ZEROSHOT_PRED_PATH,
    "static few-shot":     STATIC_PRED_PATH,
    "retrieval few-shot":  PREDICTIONS_OUTPUT_PATH,
}
available = {k: v for k, v in runs.items() if Path(v).exists()}
missing   = {k: v for k, v in runs.items() if not Path(v).exists()}
for k, v in missing.items():
    print(f"[skip] {k}: file not found ({v}) - drop its file in this dir to include it.")

common = common_indices(*available.values())
print(f"\nComparing {len(available)} run(s) on {len(common)} common rows.\n")

table = []
for name, path in available.items():
    s, _ = evaluate(path, restrict_to=common)
    table.append(dict(run=name, exact_rate=round(s["exact_rate"], 4),
                      macro_f1=round(s["macro_f1"], 4), hier_f1=round(s["h_f1"], 4),
                      over_pred=s["over_pred_rows"]))
comparison_df = pd.DataFrame(table).set_index("run")
comparison_df

## 11. Error-structure analysis — *where* it fails

An AVM code splits into **scale** (the A/B/C/D letter = roughly "how far away") and the
**object taxonomy** (what kind of thing). A useful hypothesis: the model is decent at the object
type ("it's a galaxy") but weak on scale, because cosmic distance isn't readable from pixels.
This cell decomposes accuracy that way and shows the most confused top-level object classes —
the raw material for your "LLM weaknesses" slide.

In [ ]:
from collections import Counter

def scale_letters(codes):  return {c.split(".")[0] for c in codes}
def top_objects(codes):    # e.g. C.5.1.7 -> "5" (Galaxy)
    return {c.split(".")[1] for c in codes if "." in c and len(c.split(".")) > 1}

preds = parse_prediction_file(PREDICTIONS_OUTPUT_PATH)
idxs = sorted(set(preds) & set(range(len(TRUE_AVM_CODES))))

scale_hit = obj_hit = 0
obj_confusions = Counter()
for i in idxs:
    tc, pc = TRUE_AVM_CODES[i], preds[i]
    if scale_letters(pc) & scale_letters(tc): scale_hit += 1   # any correct scale letter
    if top_objects(pc)  & top_objects(tc):    obj_hit  += 1    # any correct top-level object
    for t in top_objects(tc):
        if t not in top_objects(pc):
            for p in (top_objects(pc) or {"<none>"}):
                obj_confusions[(t, p)] += 1

n = len(idxs)
obj_name = lambda k: AVM_TAXONOMY.get(k, k)
print(f"Rows analysed: {n}")
print(f"Scale letter correct (any overlap):       {scale_hit}/{n} = {scale_hit/n:.3f}")
print(f"Top-level object correct (any overlap):    {obj_hit}/{n} = {obj_hit/n:.3f}")
print("\nMost common top-level object confusions  (true -> predicted):")
for (t, p), c in obj_confusions.most_common(10):
    print(f"  {obj_name(t):<12} -> {obj_name(p) if p!='<none>' else '<missed>':<12}  x{c}")

## 12. Compute notes & knobs to tune

**Cost ranking (highest first):**

1. **§8 inference loop** — dominant. Each prompt = `K_SHOT` example images + 1 query =
   ~11 images/forward pass vs 1 in zero-shot. Drives runtime and VRAM; main OOM risk on the 8B.
2. **§6 CLIP embedding build** — one-time, a few minutes for 266 images; cached.
3. Everything else (resize, kNN, metrics) — negligible.

**If you OOM or it's too slow:** lower `K_SHOT` (try 4–6), lower `EXAMPLE_MAX_PX` (e.g. 384),
or set `MODEL_ID` to `Qwen/Qwen3-VL-2B-Instruct`.

**Ablations worth running for the slides:**
- sweep `K_SHOT` ∈ {2, 4, 6, 10} — does more retrieval help, plateau, or hurt?
- `ADD_GLOSS` on vs off — does giving the code's *meaning* help?
- `EXCLUDE_SAME_OBJECT` off vs on — quantifies the sibling-leakage effect (report both honestly).